# GTS Spike: Validate Expression Tree Decomposition

**Goal:** Test if GTS can decompose English math word problems into parseable expression trees.

**Steps:**
1. Install MWPToolkit
2. Train GTS on MAWPS (English dataset)
3. Run inference on test problems
4. Parse prefix output to expression tree
5. Check depth and atomicity

**Runtime:** Select GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Check GPU is available
!nvidia-smi

## 1. Install MWPToolkit

In [ ]:
# Clone MWPToolkit repo
!git clone https://github.com/LYH-YF/MWPToolkit.git
%cd MWPToolkit

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install stanza  # For NLP preprocessing

In [ ]:
# Download stanza English model for tokenization
import stanza
stanza.download('en')

## 2. Check Available Datasets

In [ ]:
# List available datasets
!ls dataset/

In [ ]:
# Check MAWPS dataset (English)
import json

# Try to find MAWPS data
!find . -name "*mawps*" -o -name "*MAWPS*" 2>/dev/null | head -20

In [ ]:
# Look at a sample problem from available dataset
import os
import json

# Check what datasets exist
for dataset_dir in os.listdir('dataset'):
    print(f"Dataset: {dataset_dir}")
    dataset_path = f'dataset/{dataset_dir}'
    if os.path.isdir(dataset_path):
        for f in os.listdir(dataset_path)[:3]:
            print(f"  - {f}")

## 3. Train GTS Model

This will take ~1-2 hours on T4 GPU for MAWPS dataset.

In [ ]:
# Train GTS on MAWPS (or mawps_single if that's the name)
# Adjust dataset name based on what's available

!python run_mwptoolkit.py \
    --model=GTS \
    --dataset=mawps \
    --task_type=single_equation \
    --equation_fix=prefix \
    --k_fold=None \
    --gpu_id=0 \
    --epoch=80

## 4. Run Inference on Test Problems

**THIS IS THE KEY TEST** - Let's see what prefix expressions the model outputs.

In [ ]:
# Load the trained model and run inference
import torch
import json
import re
from pathlib import Path

# Add MWPToolkit to path
import sys
sys.path.insert(0, '.')

from mwptoolkit.config.configuration import Config
from mwptoolkit.model.Seq2Tree.gts import GTS
from mwptoolkit.utils.enum_type import SpecialTokens

# Find the trained model directory
model_dir = Path('trained_model')
for d in model_dir.iterdir():
    if d.is_dir() and 'GTS' in d.name:
        print(f"Found model: {d}")
        MODEL_PATH = d
        break

In [ ]:
# Load config and vocabularies
with open(MODEL_PATH / 'config.json') as f:
    config_dict = json.load(f)

with open(MODEL_PATH / 'input_vocab.json') as f:
    input_vocab = json.load(f)

with open(MODEL_PATH / 'output_vocab.json') as f:
    output_vocab = json.load(f)

print("Config loaded")
print(f"Input vocab size: {len(input_vocab['in_idx2word'])}")
print(f"Output vocab: {output_vocab['out_idx2symbol']}")

In [ ]:
# Create word to index mapping
word2idx = {word: idx for idx, word in enumerate(input_vocab['in_idx2word'])}
idx2symbol = output_vocab['out_idx2symbol']

# Special tokens
PAD_IDX = word2idx.get('<PAD>', 0)
SOS_IDX = word2idx.get('<SOS>', 1)
EOS_IDX = word2idx.get('<EOS>', 2)
UNK_IDX = word2idx.get('<UNK>', 3)

print(f"PAD={PAD_IDX}, SOS={SOS_IDX}, EOS={EOS_IDX}, UNK={UNK_IDX}")

In [ ]:
# Helper function to preprocess problem text
def preprocess_problem(text):
    """
    Preprocess problem text:
    1. Extract numbers and replace with NUM
    2. Tokenize
    3. Convert to indices
    """
    # Extract numbers
    numbers = []
    def replace_num(match):
        num = match.group()
        numbers.append(float(num) if '.' in num else int(num))
        return 'NUM'
    
    # Replace numbers with NUM token
    text_normalized = re.sub(r'\d+\.?\d*', replace_num, text)
    
    # Simple tokenization (split on spaces and punctuation)
    tokens = re.findall(r"[\w']+|[.,!?;]", text_normalized)
    
    # Convert to indices
    indices = [word2idx.get(t, UNK_IDX) for t in tokens]
    
    return {
        'original': text,
        'normalized': text_normalized,
        'tokens': tokens,
        'indices': indices,
        'numbers': numbers
    }

# Test preprocessing
test = preprocess_problem("John has 5 apples. Mary gives him 3 more. How many does he have?")
print(f"Original: {test['original']}")
print(f"Normalized: {test['normalized']}")
print(f"Tokens: {test['tokens']}")
print(f"Numbers: {test['numbers']}")

In [ ]:
# Load the actual model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model weights
model_state = torch.load(MODEL_PATH / 'model.pth', map_location=device)
print(f"Model state keys: {list(model_state.keys())[:10]}...")

In [ ]:
# Use MWPToolkit's test functionality to run inference
# This is easier than manually reconstructing the model

from mwptoolkit.quick_start import test_with_trained_model

# Check if this function exists and how to use it
help(test_with_trained_model)

In [ ]:
# Alternative: Look at what the test set results look like
# Check if there are any result files from training
!find . -name "*.json" -path "*/result/*" 2>/dev/null | head -10
!find . -name "*result*" -type f 2>/dev/null | head -10

In [ ]:
# Let's look at the test data to see example input/output pairs
!ls dataset/mawps/

In [ ]:
# Load and examine some test examples with their equations
import json

# Find test data file
test_files = list(Path('dataset/mawps').glob('*.json'))
print(f"Found files: {test_files}")

# Load first file and show examples
if test_files:
    with open(test_files[0]) as f:
        data = json.load(f)
    
    print(f"\nNumber of examples: {len(data)}")
    print("\n" + "="*60)
    print("SAMPLE PROBLEMS WITH EQUATIONS:")
    print("="*60)
    
    for i, ex in enumerate(data[:10]):
        print(f"\n--- Example {i+1} ---")
        print(f"Problem: {ex.get('original_text', ex.get('text', 'N/A'))}")
        print(f"Equation: {ex.get('equation', ex.get('prefix', 'N/A'))}")
        print(f"Answer: {ex.get('answer', ex.get('ans', 'N/A'))}")

In [ ]:
# Run the model on the test set and capture outputs
# This uses MWPToolkit's built-in evaluation

!python run_mwptoolkit.py \
    --model=GTS \
    --dataset=mawps \
    --task_type=single_equation \
    --equation_fix=prefix \
    --gpu_id=0 \
    --test_only=True \
    --trained_model_dir=trained_model/GTS-mawps 2>&1 | tail -50

## 5. Parse Prefix Output to Expression Tree

In [ ]:
# Expression tree parser (same as our planned integration)

from dataclasses import dataclass
from typing import Optional
from enum import Enum

class NodeType(Enum):
    OPERATOR = "operator"
    NUMBER = "number"
    VARIABLE = "variable"

@dataclass
class ExprNode:
    type: NodeType
    value: str
    left: Optional["ExprNode"] = None
    right: Optional["ExprNode"] = None
    
    @property
    def depth(self) -> int:
        if self.type != NodeType.OPERATOR:
            return 0
        left_depth = self.left.depth if self.left else 0
        right_depth = self.right.depth if self.right else 0
        return 1 + max(left_depth, right_depth)
    
    @property
    def is_atomic(self) -> bool:
        return self.depth <= 1
    
    def __repr__(self):
        if self.type != NodeType.OPERATOR:
            return self.value
        return f"({self.value} {self.left} {self.right})"

def parse_prefix(prefix_str: str) -> ExprNode:
    """Parse prefix notation to expression tree."""
    tokens = prefix_str.replace('(', '').replace(')', '').split()
    idx = 0
    
    def parse() -> ExprNode:
        nonlocal idx
        if idx >= len(tokens):
            raise ValueError("Unexpected end of expression")
        
        token = tokens[idx]
        idx += 1
        
        # Operators
        if token in ['+', '-', '*', '/', '^', 'ADD', 'SUB', 'MUL', 'DIV']:
            # Normalize operator names
            op_map = {'ADD': '+', 'SUB': '-', 'MUL': '*', 'DIV': '/'}
            op = op_map.get(token, token)
            return ExprNode(
                type=NodeType.OPERATOR,
                value=op,
                left=parse(),
                right=parse()
            )
        # Variables (NUM_0, NUM_1, etc.)
        elif token.startswith('NUM') or token.startswith('temp_') or token == 'x':
            return ExprNode(type=NodeType.VARIABLE, value=token)
        # Numbers
        else:
            try:
                float(token)
                return ExprNode(type=NodeType.NUMBER, value=token)
            except ValueError:
                return ExprNode(type=NodeType.VARIABLE, value=token)
    
    return parse()

# Test the parser with GTS-style prefix expressions
test_prefixes = [
    "+ NUM_0 NUM_1",           # Simple addition
    "- NUM_0 NUM_1",           # Simple subtraction  
    "* NUM_0 NUM_1",           # Simple multiplication
    "/ NUM_0 NUM_1",           # Simple division
    "- + NUM_0 NUM_1 NUM_2",   # (NUM_0 + NUM_1) - NUM_2, depth 2
    "+ * NUM_0 NUM_1 NUM_2",   # (NUM_0 * NUM_1) + NUM_2, depth 2
]

print("Testing prefix parser with GTS-style expressions:")
print("=" * 60)
for prefix in test_prefixes:
    try:
        tree = parse_prefix(prefix)
        print(f"\nPrefix: {prefix}")
        print(f"Tree:   {tree}")
        print(f"Depth:  {tree.depth}")
        print(f"Atomic: {tree.is_atomic}")
    except Exception as e:
        print(f"\nPrefix: {prefix}")
        print(f"ERROR: {e}")

## 6. Analyze Dataset Equations for Depth Distribution

In [ ]:
# Analyze the depth distribution of equations in MAWPS
# This tells us how many are atomic (depth 1) vs need decomposition

depths = []
errors = []

for ex in data:
    equation = ex.get('prefix', ex.get('equation', ''))
    if equation:
        try:
            # Clean up equation format if needed
            eq_clean = equation.replace('[', '').replace(']', '').replace(',', ' ')
            tree = parse_prefix(eq_clean)
            depths.append(tree.depth)
        except Exception as e:
            errors.append((equation, str(e)))

print("DEPTH DISTRIBUTION:")
print("=" * 40)
from collections import Counter
depth_counts = Counter(depths)
for d in sorted(depth_counts.keys()):
    pct = 100 * depth_counts[d] / len(depths)
    print(f"Depth {d}: {depth_counts[d]:4d} ({pct:5.1f}%)")

print(f"\nTotal parsed: {len(depths)}")
print(f"Parse errors: {len(errors)}")

atomic_pct = 100 * depth_counts.get(1, 0) / len(depths) if depths else 0
print(f"\nAtomic (depth 1): {atomic_pct:.1f}%")
print(f"Need decomposition (depth > 1): {100 - atomic_pct:.1f}%")

In [ ]:
# Show some examples that need decomposition (depth > 1)
print("\nEXAMPLES NEEDING DECOMPOSITION (depth > 1):")
print("=" * 60)

shown = 0
for ex in data:
    equation = ex.get('prefix', ex.get('equation', ''))
    if equation:
        try:
            eq_clean = equation.replace('[', '').replace(']', '').replace(',', ' ')
            tree = parse_prefix(eq_clean)
            if tree.depth > 1:
                print(f"\nProblem: {ex.get('original_text', ex.get('text', 'N/A'))[:100]}...")
                print(f"Equation: {equation}")
                print(f"Tree: {tree}")
                print(f"Depth: {tree.depth}")
                shown += 1
                if shown >= 5:
                    break
        except:
            pass

## 7. Results Summary

In [ ]:
# FILL THIS IN AFTER RUNNING THE NOTEBOOK

results = {
    "installation": "SUCCESS",  # or FAILED
    "training_completed": "YES",  # or NO
    "training_epochs": 80,
    "equation_accuracy": "XX%",  # Fill from training output
    "value_accuracy": "XX%",  # Fill from training output
    "total_problems": len(data) if 'data' in dir() else 0,
    "atomic_percentage": f"{atomic_pct:.1f}%" if 'atomic_pct' in dir() else "N/A",
    "prefix_format_works": "YES",  # Can we parse the prefix output?
    "issues": [],
    "recommendation": "PROCEED / PIVOT / NEEDS_MORE_TESTING"
}

print("=" * 60)
print("GTS SPIKE RESULTS")
print("=" * 60)
for k, v in results.items():
    print(f"{k}: {v}")

print("\n" + "=" * 60)
print("DECISION:")
print("=" * 60)
print("""
If accuracy > 70% and prefix format parses correctly:
  → PROCEED with GTS integration
  
If accuracy < 50% or major parsing issues:
  → PIVOT to alternative (small LLM or different model)
  
If unclear:
  → NEEDS_MORE_TESTING (try more problems, check edge cases)
""")

## 8. Save Trained Model (if successful)

In [ ]:
# Zip trained model for download
!zip -r gts_mawps_trained.zip trained_model/

# In Colab, this will let you download
from google.colab import files
files.download('gts_mawps_trained.zip')